# PEMS-SF EDA (GPU 版, v4)
目标：在 GPU 可用时尽量使用 GPU（CuPy/RAPIDS 风格）进行数值聚合，保持与 v3 相同/更优的功能与结果；在无 GPU 或未安装 CuPy 时自动回退到 CPU(Numpy)。

In [ ]:
# 初始化依赖与 GPU/CPU 选择
import pathlib, os, re, math, sys, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8')
DATA_DIR = pathlib.Path('data')
# 优先使用 CuPy 作为 GPU 数组后端；不可用则回落到 Numpy
try:
    import cupy as cp
    # 检测设备
    _gpu_ok = (cp.cuda.runtime.getDeviceCount() > 0)
    xp = cp if _gpu_ok else np
except Exception:
    cp = None
    _gpu_ok = False
    xp = np
def to_cpu(a):
    """将后端数组转为 CPU numpy 数组。"""
    if 'cupy' in str(type(a)) or (cp is not None and isinstance(a, cp.ndarray)):
        return cp.asnumpy(a)
    return np.asarray(a)
print('Array backend ->', 'CuPy(GPU)' if _gpu_ok else 'NumPy(CPU)')

In [ ]:
# 字体与绘图文本映射
from matplotlib import rcParams
rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'Noto Sans CJK SC']
rcParams['axes.unicode_minus'] = False
plot_texts = {
    'train_label_dist_title': 'Train label distribution',
    'train_daily_mean_boxplot': 'Train: daily mean occupancy by weekday label',
    'avg_daily_profile_by_label': 'Average daily occupancy profile (by label)',
    'time_slot_xlabel': 'Time slot (10-min units, 144 slots)',
    'avg_occupancy_ylabel': 'Average occupancy',
    'station_weekday_profile_title': 'Station {sid}: Mean occupancy per time slot by weekday (1-7)',
    'station_daily_mean_title': 'Station {sid}: Daily mean occupancy by weekday label',
    'station_plot_xlabel': 'Weekday label (1=Mon ... 7=Sun)',
    'station_plot_ylabel': 'Daily mean occupancy',
}
def L(key, **kw):
    s = plot_texts.get(key, key)
    return s.format(**kw)

## 数据/标签读取与解析 (与 v3 一致)

In [ ]:
# 读取标签文件
from collections import Counter
def load_label_file(path: pathlib.Path):
    txt = path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    nums = [int(x) for x in txt.split() if re.match(r'^-?\d+$', x)]
    return np.array(nums, dtype=int) if nums else None
labels_train = load_label_file(DATA_DIR/'PEMS_trainlabels') if (DATA_DIR/'PEMS_trainlabels').exists() else None
labels_test  = load_label_file(DATA_DIR/'PEMS_testlabels')  if (DATA_DIR/'PEMS_testlabels').exists()  else None
labels_summary = {}
for name, arr in [('train', labels_train), ('test', labels_test)]:
    if arr is not None:
        labels_summary[name] = {
            'count': int(arr.size),
            'unique': np.unique(arr).tolist(),
            'value_counts': dict(Counter(arr)),
            'min': int(arr.min()),
            'max': int(arr.max())
        }
labels_summary

In [ ]:
# 解析 stations_list (单行包含 963 个 ID)
stations_path = DATA_DIR/'stations_list'
if stations_path.exists():
    raw = stations_path.read_text(encoding='utf-8', errors='ignore').strip().strip('[]')
    station_ids = [int(x) for x in raw.split() if re.match(r'^\d+$', x)]
    print('站点数量解析:', len(station_ids))
else:
    station_ids = []
    print('stations_list 不存在')
assert len(station_ids) in (0, 963), f'解析到的站点数量异常: {len(station_ids)}'

## 单行日矩阵解析与流式读取

In [ ]:
def parse_day_matrix(line: str, expected_rows=963, expected_cols=144):
    line = line.strip()
    if line.startswith('[') and line.endswith(']'): line = line[1:-1]
    rows = [r.strip() for r in line.split(';') if r.strip()]
    data = []
    for r in rows:
        nums = [float(x) for x in r.split() if re.match(r'^-?\d+(\.\d+)?$', x)]
        if nums: data.append(nums)
    if not data: return None
    lens = [len(rr) for rr in data]
    maxc = max(lens)
    if len(set(lens)) == 1:
        arr = np.array(data, dtype=float)
    else:
        arr = np.full((len(data), maxc), np.nan, dtype=float)
        for i, rr in enumerate(data): arr[i, :len(rr)] = rr
    return arr

In [ ]:
def iter_day_matrices_optimized(path: pathlib.Path, limit=None):
    with path.open('r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit: break
            mat = parse_day_matrix(line)
            if mat is None: continue
            yield mat  # CPU numpy 矩阵，计算阶段再转 GPU

## 全局统计与有效性检查（GPU 加速聚合）

In [ ]:
train_path = DATA_DIR/'PEMS_train'
test_path  = DATA_DIR/'PEMS_test'
expected_shape = (963, 144)
def stream_global_gpu(path: pathlib.Path, labels=None, sample_per_day=4096):
    total_sum = 0.0
    total_sumsq = 0.0
    total_count = 0
    gmin = math.inf
    gmax = -math.inf
    sample = []
    rng = np.random.default_rng(2025)
    for day_i, mat_cpu in enumerate(iter_day_matrices_optimized(path)):
        # 转 GPU/CPU 通用后端
        arr = xp.asarray(mat_cpu)
        # 基本范围
        dmin = float(xp.nanmin(arr))
        dmax = float(xp.nanmax(arr))
        if dmin < gmin: gmin = dmin
        if dmax > gmax: gmax = dmax
        # 求和与平方和
        c = arr.size - int(xp.isnan(arr).sum())
        s = float(xp.nansum(arr))
        ss = float(xp.nansum(arr*arr))
        total_sum += s; total_sumsq += ss; total_count += c
        # 近似抽样（均匀下标抽样）
        if sample_per_day > 0:
            n = int(arr.size)
            k = min(sample_per_day, n)
            # 在 CPU 生成索引，避免跨设备随机数不一致
            idx = rng.choice(n, size=k, replace=False)
            flat = arr.ravel()
            pick = to_cpu(flat[idx])
            # 过滤 NaN
            pick = pick[~np.isnan(pick)]
            if pick.size: sample.append(pick)
    if total_count == 0:
        return {'value_count': 0}
    mean = total_sum / total_count
    var = max(total_sumsq / total_count - mean*mean, 0.0)
    std = math.sqrt(var)
    sample_all = np.concatenate(sample) if len(sample) else np.array([])
    quantiles = {}
    if sample_all.size:
        qs = [0,0.01,0.05,0.25,0.5,0.75,0.95,0.99,1]
        quantiles = {q: float(np.quantile(sample_all, q)) for q in qs}
    return {
        'value_count': int(total_count),
        'global_min': float(gmin),
        'global_max': float(gmax),
        'mean': float(mean),
        'std': float(std),
        'approx_quantiles': quantiles
    }
train_stats_preview = stream_global_gpu(train_path, labels_train)
test_stats_preview  = stream_global_gpu(test_path,  labels_test)
train_stats_preview, test_stats_preview

In [ ]:
def occupancy_validity_gpu(path: pathlib.Path, max_days=None):
    out_of_range = 0
    nan_count = 0
    total_days = 0
    for day_i, mat_cpu in enumerate(iter_day_matrices_optimized(path, limit=max_days)):
        total_days += 1
        arr = xp.asarray(mat_cpu)
        nan_count += int(xp.isnan(arr).sum())
        oor = xp.sum((arr < 0) | (arr > 1))
        out_of_range += int(to_cpu(oor))
    return {'days_checked': total_days, 'nan_values': int(nan_count), 'out_of_range_values': int(out_of_range)}
validity_train_preview = occupancy_validity_gpu(train_path, max_days=10)
validity_train_preview

## 标签分布与日级统计/可视化（与 v3 对齐）

In [ ]:
def daily_stats_gpu(path: pathlib.Path, labels=None):
    rows = []
    for day_i, mat_cpu in enumerate(iter_day_matrices_optimized(path)):
        arr = xp.asarray(mat_cpu).ravel()
        vmin = float(xp.nanmin(arr))
        vmax = float(xp.nanmax(arr))
        vmean = float(xp.nanmean(arr))
        vmed = float(xp.nanmedian(arr))
        vstd = float(xp.nanstd(arr))
        row = {'day_index': day_i, 'min': vmin, 'max': vmax, 'mean': vmean, 'median': vmed, 'std': vstd}
        if labels is not None and day_i < len(labels): row['label'] = int(labels[day_i])
        rows.append(row)
    return pd.DataFrame(rows)
daily_train_df = daily_stats_gpu(train_path, labels_train)
daily_train_df.head()

In [ ]:
# 标签分布
if labels_train is not None:
    vc = pd.Series(labels_train).value_counts().sort_index()
    plt.figure(figsize=(6,3))
    sns.barplot(x=vc.index, y=vc.values, palette='tab10')
    plt.title(L('train_label_dist_title')); plt.xlabel('Label'); plt.ylabel('Count'); plt.tight_layout(); plt.show()
# 日均占有率箱线图
plt.figure(figsize=(6,4))
sns.boxplot(data=daily_train_df, x='label', y='mean')
plt.title(L('train_daily_mean_boxplot')); plt.xlabel('Label(weekdays)'); plt.ylabel('Daily Mean Occupancy'); plt.tight_layout(); plt.show()

## 标签均值日内曲线（GPU 聚合）

In [ ]:
def label_mean_profiles_gpu(path: pathlib.Path, labels):
    acc_sum = {}  # lb -> 144 向量
    acc_cnt = {}
    for day_i, mat_cpu in enumerate(iter_day_matrices_optimized(path)):
        if day_i >= len(labels): break
        lb = int(labels[day_i])
        arr = xp.asarray(mat_cpu)
        prof = xp.nanmean(arr, axis=0)  # 144
        if lb not in acc_sum:
            acc_sum[lb] = xp.zeros_like(prof)
            acc_cnt[lb] = 0
        acc_sum[lb] += prof; acc_cnt[lb] += 1
    out = {}
    for lb in sorted(acc_sum.keys()):
        out[lb] = to_cpu(acc_sum[lb] / max(acc_cnt[lb], 1))
    return out
profiles = label_mean_profiles_gpu(train_path, labels_train) if labels_train is not None else {}
plt.figure(figsize=(10,5))
for lb in sorted(profiles.keys()): plt.plot(profiles[lb], label=f'Label {lb}')
plt.title(L('avg_daily_profile_by_label'))
plt.xlabel(L('time_slot_xlabel')); plt.ylabel(L('avg_occupancy_ylabel')); plt.legend(ncol=2); plt.tight_layout(); plt.show()

## 站点级高效可视化（GPU 版聚合，963 站点）

In [ ]:
assert labels_train is not None, '训练标签未加载'
assert len(station_ids) == 963, 'stations_list 未解析或数量不为 963'
sensors, time_slots = 963, 144
# GPU 累加器：(sensor, label, slot) 汇总 + 次数
profiles_sum = xp.zeros((sensors, 7, time_slots), dtype=float)
profiles_cnt = xp.zeros((sensors, 7), dtype=int)
daily_rows = []  # 用于箱线图：每站点每天均值
for day_i, mat_cpu in enumerate(iter_day_matrices_optimized(train_path)):
    if day_i >= len(labels_train): break
    lb = int(labels_train[day_i]) - 1  # 0..6
    arr = xp.asarray(mat_cpu)  # (963,144)
    # 站点日均值
    day_means = xp.nanmean(arr, axis=1)  # (963,)
    day_means_cpu = to_cpu(day_means)
    for ridx in range(sensors):
        daily_rows.append({'station_id': int(station_ids[ridx]), 'label': lb+1, 'daily_mean': float(day_means_cpu[ridx])})
    # 周内时间槽均值累加
    profiles_sum[:, lb, :] += arr
    profiles_cnt[:, lb] += 1
station_label_daily_means = pd.DataFrame(daily_rows)
# 展开为长表用于折线图（每站点、每星期、每时间槽 均值）
rows = []
cnt_cpu = to_cpu(profiles_cnt)
sum_cpu = to_cpu(profiles_sum)
for ridx in range(sensors):
    sid = int(station_ids[ridx])
    for lb in range(7):
        c = int(cnt_cpu[ridx, lb])
        if c == 0: continue
        mean_vec = sum_cpu[ridx, lb, :] / c
        for t in range(time_slots):
            rows.append({'station_id': sid, 'label': lb+1, 'slot': t, 'mean_occupancy': float(mean_vec[t])})
station_weekday_time_profiles_long = pd.DataFrame(rows)
station_label_daily_means.head(), station_weekday_time_profiles_long.head()

In [ ]:
# 分页保存箱线图
def save_station_boxplots_paginated(df: pd.DataFrame, per_page=24, ncols=3, height_per_row=2.5, out_dir='Stage1-figure/box', dpi=150):
    import math
    out = pathlib.Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    station_ids_sorted = sorted(df['station_id'].unique().tolist())
    n_pages = math.ceil(len(station_ids_sorted) / per_page)
    for page in range(n_pages):
        start = page * per_page; end = min(start + per_page, len(station_ids_sorted))
        subset_ids = station_ids_sorted[start:end]
        nrows = math.ceil(len(subset_ids) / ncols)
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 4, nrows * height_per_row))
        axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
        for ax, sid in zip(axes, subset_ids):
            sub = df[df['station_id'] == sid]
            sns.boxplot(data=sub, x='label', y='daily_mean', ax=ax, color='#88c')
            sns.stripplot(data=sub, x='label', y='daily_mean', ax=ax, color='#333', alpha=0.4, jitter=0.15)
            ax.set_title(L('station_daily_mean_title', sid=sid))
            ax.set_xlabel(L('station_plot_xlabel'))
            ax.set_ylabel(L('station_plot_ylabel'))
        for ax in axes[len(subset_ids):]: ax.axis('off')
        plt.tight_layout()
        plt.savefig(out / f'page_{page+1}.png', dpi=dpi)
        plt.close(fig)
save_station_boxplots_paginated(station_label_daily_means, per_page=24, ncols=3, height_per_row=2.5, out_dir='Stage1-figure/box', dpi=150)

In [ ]:
# 分页保存周内折线图
def save_station_weekday_lines_paginated(df_long: pd.DataFrame, per_page=12, out_dir='Stage1-figure/line', dpi=150):
    out = pathlib.Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    station_ids_sorted = sorted(df_long['station_id'].unique().tolist())
    for page_i in range(0, len(station_ids_sorted), per_page):
        subset_ids = station_ids_sorted[page_i: page_i+per_page]
        n = len(subset_ids)
        fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(12, 3*n))
        if n == 1: axes = [axes]
        for ax, sid in zip(axes, subset_ids):
            sub = df_long[df_long['station_id'] == sid]
            for lb in range(1,8):
                s2 = sub[sub['label'] == lb].sort_values('slot')
                ax.plot(s2['slot'], s2['mean_occupancy'], label=f'{lb}')
            ax.set_title(L('station_weekday_profile_title', sid=sid))
            ax.set_xlabel(L('time_slot_xlabel'))
            ax.set_ylabel(L('avg_occupancy_ylabel'))
            ax.legend(title='Weekday', ncol=7, fontsize=8, title_fontsize=9)
        plt.tight_layout()
        plt.savefig(out / f'page_{page_i//per_page + 1}.png', dpi=dpi)
        plt.close(fig)
save_station_weekday_lines_paginated(station_weekday_time_profiles_long, per_page=12, out_dir='Stage1-figure/line', dpi=150)